# Ensemble Regression


In [ ]:
# ==============================================================================
# --- GLOBAL CONFIGURATION & PARAMETERS ---
# ==============================================================================
# System-wide configuration and adjustable parameters.

# --- 1. Data Source and Descriptor Selection ---
# Engineered workbook produced by the companion notebook in this release.
EXCEL_FILE_PATH = 'feature_engineering_output/il_lumo_engineered_data.xlsx'

# Fixed schema: the first column is an identifier, descriptors are in the middle,
# and the final column is always the single regression target (LUMO).
DECIMAL_PLACES = 4  # Round all descriptors and the target before fitting.

# --- 2. Cross-Validation Strategy ---
CV_N_SPLITS = 10         # Use 5 for small datasets; use 10 when sample size supports stable folds.
CV_SHUFFLE = True        # Shuffle i.i.d. samples; set False only for ordered or time-series data.
CV_RANDOM_STATE = 100    # Keep fixed for reproducibility; change only for a sensitivity analysis.

# --- 3. Bayesian Optimization Configuration ---
N_ITER_BAYESIAN = 30     # Increase for a larger search budget; reduce for rapid exploratory runs.
BAYES_OPTIMIZER_KWARGS = {"base_estimator": "GP", "acq_func": "EI"}  # GP/EI balances exploration and exploitation; retain for SI-consistent tuning.

# --- 4. Global Model Initialization ---
DEFAULT_MODEL_RANDOM_STATE = 0  # Keep fixed for reproducible model comparisons and exports.

# --- 5. Active Model Selection ---
# All seven candidates are enabled by default; comment out one entry only for an intentional ablation.
ENABLED_MODELS = [
    'XGBR',   # XGBoost Regressor
    'RF',     # Random Forest Regressor
    'GBRT',   # Gradient Boosting Regressor
    'HGBR',   # Histogram-based Gradient Boosting Regressor
    'ETR',    # Extra Trees Regressor
    'CBR',    # CatBoost Regressor
    'LGBM',   # LightGBM Regressor
]


# ==============================================================================
# --- HYPERPARAMETER SEARCH SPACES ---
# ==============================================================================
# Definition of the search space for Bayesian optimization for each estimator.
from skopt.space import Real, Integer, Categorical

parameter_XGBR = {
    'n_estimators': Integer(10, 500),       # More trees reduce variance; lower for fast pilots.
    'learning_rate': Real(0.01, 0.5, prior='log-uniform'),  # Lower values are safer for noisy or small data.
    'max_depth': Integer(1, 10),            # Reduce the upper bound when overfitting is evident.
    'subsample': Real(0.5, 0.9, prior='uniform'),  # Lower values add regularization through row sampling.
    'colsample_bytree': Real(0.5, 1.0, prior='uniform'),  # Lower values increase feature diversity.
    'reg_alpha': Real(0.1, 1.0, prior='log-uniform'),  # Increase to suppress weak or noisy effects.
    'reg_lambda': Real(0.1, 10.0, prior='log-uniform')  # Increase to stabilize correlated descriptors.
}
parameter_RF = {
    'n_estimators': Integer(10, 200),       # More trees improve stability; lower for quick screening.
    'max_depth': Integer(1, 20),            # Lower depths reduce overfitting on small datasets.
    'max_features': Categorical(['sqrt', 'log2', 1.0]),  # Fewer features increase ensemble diversity.
    'min_samples_leaf': Integer(1, 10),     # Increase to smooth noisy or sparse responses.
    'min_samples_split': Integer(2, 10)     # Increase to suppress noise-driven branches.
}
parameter_CBR = {
    'iterations': Integer(10, 500),         # More rounds improve fit; lower when validation begins to decline.
    'learning_rate': Real(0.01, 0.5, prior='log-uniform'),  # Lower rates are safer for small datasets.
    'depth': Integer(1, 16),                # Reduce depth when validation variance is high.
    'l2_leaf_reg': Real(0.1, 10.0, prior='log-uniform'),  # Increase to regularize complex trees.
    'subsample': Real(0.5, 0.9, prior='uniform'),  # Lower values can improve generalization.
    'rsm': Real(0.5, 1.0, prior='uniform')   # Lower values reduce descriptor co-adaptation.
}
parameter_LGBM = {
    'n_estimators': Integer(10, 500),       # More rounds improve fit; reduce for rapid screening.
    'learning_rate': Real(0.01, 0.8, prior='log-uniform'),  # Lower values reduce aggressive fitting.
    'max_depth': Integer(1, 10),            # Lower depths are safer for small samples.
    'num_leaves': Integer(5, 50),           # Fewer leaves reduce nonlinear overfitting.
    'subsample': Real(0.5, 0.9, prior='uniform'),  # Lower values improve robustness through row sampling.
    'colsample_bytree': Real(0.5, 1.0, prior='uniform'),  # Lower values limit feature co-adaptation.
    'reg_alpha': Real(0.1, 10.0, prior='log-uniform'),  # Increase to promote sparse feature use.
    'reg_lambda': Real(0.1, 10.0, prior='log-uniform')  # Increase to stabilize correlated inputs.
}
parameter_GBRT = {
    'n_estimators': Integer(10, 500),       # More rounds improve fit; reduce when overfitting appears.
    'learning_rate': Real(0.01, 0.5, prior='log-uniform'),  # Lower values suit noisy data.
    'max_depth': Integer(1, 10),            # Lower depths limit interaction complexity.
    'max_features': Categorical(['sqrt', 'log2', 1.0]),  # Fewer features improve model diversity.
    'min_samples_split': Integer(2, 10),    # Increase to suppress noise-driven splits.
    'min_samples_leaf': Integer(1, 10),     # Increase to smooth local fluctuations.
    'subsample': Real(0.5, 0.9, prior='uniform')  # Lower values can improve generalization.
}
parameter_HGBR = {
    'learning_rate': Real(0.01, 0.5, prior='log-uniform'),  # Lower values reduce overfitting risk.
    'max_iter': Integer(10, 500),            # More iterations increase capacity; reduce for small data.
    'max_depth': Integer(1, 10),             # Lower depths are preferable for small datasets.
    'min_samples_leaf': Integer(2, 10),      # Increase to reduce sensitivity to noise.
    'l2_regularization': Real(0.1, 10.0, prior='log-uniform')  # Increase to stabilize weak signals.
}
parameter_ETR = {
    'n_estimators': Integer(10, 200),        # More randomized trees stabilize averaging; lower for quick runs.
    'max_depth': Integer(1, 10),             # Lower depths reduce variance on small datasets.
    'min_samples_split': Integer(2, 10),     # Increase to reduce spurious branching.
    'min_samples_leaf': Integer(1, 10),      # Increase to smooth noisy responses.
    'max_features': Categorical(['sqrt', 'log2', 1.0])  # Fewer features increase tree diversity.
}

# ==============================================================================
# --- MAIN EXECUTION PIPELINE ---
# ==============================================================================

# --- 1. Environment Initialization ---
import time
import pandas as pd
import numpy as np
import re
import xgboost as XGB
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold
from skopt import BayesSearchCV

print("--- Pipeline Initialized ---")

try:
    from catboost import CatBoostRegressor
    catboost_available = True
except ImportError:
    catboost_available = False
try:
    from lightgbm import LGBMRegressor
    lightgbm_available = True
except ImportError:
    lightgbm_available = False
print("Dependencies verified.")


# --- 2. Data Ingestion and Preprocessing ---
X, Y = pd.DataFrame(), pd.Series()
try:
    print(f"\nImporting dataset: {EXCEL_FILE_PATH}")
    df = pd.read_excel(EXCEL_FILE_PATH)
    print("Dataset imported successfully.")

    if df.shape[1] < 3:
        raise ValueError("CRITICAL: The workbook must contain an identifier, at least one descriptor, and a final target column.")
    numeric_cols = df.columns[1:]
    df.loc[:, numeric_cols] = df.loc[:, numeric_cols].apply(pd.to_numeric, errors='coerce').round(DECIMAL_PLACES)
    X = df.iloc[:, 1:-1]
    Y = df.iloc[:, -1]
    print(f"Feature/Target extraction complete. X shape={X.shape}, Y shape={Y.shape} (Target: '{Y.name}')")

    print("\nSanitizing feature column names for estimator compatibility...")
    X.columns = [re.sub(r'\[|\]|<', '_', col) for col in X.columns]
    print("Column sanitization complete.")

except FileNotFoundError:
    print(f"\n!!! CRITICAL: File '{EXCEL_FILE_PATH}' not found. Check configuration.")
    exit()
except Exception as e:
    print(f"\n!!! CRITICAL: Data ingestion failure: {e}")
    exit()

# --- 3. Model Instantiation and Validation Strategy ---
cross_Valid = KFold(n_splits=CV_N_SPLITS, shuffle=CV_SHUFFLE, random_state=CV_RANDOM_STATE)

# Registry of all supported estimators
ALL_POSSIBLE_ESTIMATORS = {
    'XGBR': XGB.XGBRegressor(random_state=DEFAULT_MODEL_RANDOM_STATE, objective='reg:squarederror'),
    'RF': RandomForestRegressor(random_state=DEFAULT_MODEL_RANDOM_STATE),
    'GBRT': GradientBoostingRegressor(random_state=DEFAULT_MODEL_RANDOM_STATE),
    'HGBR': HistGradientBoostingRegressor(random_state=DEFAULT_MODEL_RANDOM_STATE),
    'ETR': ExtraTreesRegressor(random_state=DEFAULT_MODEL_RANDOM_STATE)
}
if catboost_available:
    ALL_POSSIBLE_ESTIMATORS['CBR'] = CatBoostRegressor(verbose=False, random_state=DEFAULT_MODEL_RANDOM_STATE, allow_writing_files=False)
else:
    print("Info: CatBoost module not found; related configurations ignored.")

if lightgbm_available:
    ALL_POSSIBLE_ESTIMATORS['LGBM'] = LGBMRegressor(random_state=DEFAULT_MODEL_RANDOM_STATE, verbosity=-1, objective='regression')
else:
    print("Info: LightGBM module not found; related configurations ignored.")

# Filter active estimators based on user configuration
estimators_for_bayes = {name: ALL_POSSIBLE_ESTIMATORS[name]
                        for name in ENABLED_MODELS
                        if name in ALL_POSSIBLE_ESTIMATORS}

params_mapping = {
    'XGBR': parameter_XGBR, 'RF': parameter_RF, 'GBRT': parameter_GBRT,
    'HGBR': parameter_HGBR, 'ETR': parameter_ETR
}
if catboost_available: params_mapping['CBR'] = parameter_CBR
if lightgbm_available: params_mapping['LGBM'] = parameter_LGBM

if not estimators_for_bayes:
    print("\n!!! CRITICAL: No active models configured. Verify 'ENABLED_MODELS' in configuration.")
    exit()

print(f"\nPipeline ready. Starting optimization ({cross_Valid.get_n_splits()}-fold CV) for models: {list(estimators_for_bayes.keys())}")


# --- 4. Bayesian Optimization Loop ---
grid_searches = {}
print(f"\nInitiating BayesSearchCV (Iterations={N_ITER_BAYESIAN})...")

for name, estimator in estimators_for_bayes.items():
    start_time = time.time()
    print(f"\n--- Optimizing {name} ---")
    if name not in params_mapping:
        print(f"Warning: No hyperparameter space defined for {name}. Skipping.")
        grid_searches[name] = None
        continue

    bayes_search = BayesSearchCV(
        estimator=estimator, search_spaces=params_mapping[name],
        n_iter=N_ITER_BAYESIAN, scoring='r2', cv=cross_Valid,
        n_jobs=-1, random_state=DEFAULT_MODEL_RANDOM_STATE,
        optimizer_kwargs=BAYES_OPTIMIZER_KWARGS, verbose=1
    )
    try:
        bayes_search.fit(X, Y)
        duration = time.time() - start_time
        grid_searches[name] = bayes_search
        print(f"--- {name} Optimization Complete ---")
        print(f"  Best Score (CV R²): {bayes_search.best_score_:.4f}")
        print(f"  Best Parameters: {dict(bayes_search.best_params_)}")
        print(f"  Runtime: {duration:.2f} s")
    except Exception as e:
        duration = time.time() - start_time
        print(f"\n!!! ERROR: Optimization failed for {name}: {e}")
        print(f"  Runtime before failure: {duration:.2f} s")
        grid_searches[name] = None

# ==============================================================================
# --- 5. RESULTS SUMMARY ---
# ==============================================================================
print("\n\n==============================================================================")
print("--- OPTIMIZATION REPORT ---")
print("==============================================================================")

# Iterate and report results
for name, search_result in grid_searches.items():
    print(f"\n--- Model: {name} ---")
    if search_result:
        best_score = search_result.best_score_
        best_params = dict(search_result.best_params_)

        print(f"  Best R² Score (CV): {best_score:.4f}")
        print("  Optimal Hyperparameters:")
        for param, value in best_params.items():
            if isinstance(value, float):
                print(f"    - {param}: {value:.6f}")
            else:
                print(f"    - {param}: {value}")
    else:
        print("  Status: Failed or Skipped.")
print("\n--- Pipeline Execution Finished ---")

Model stacking, robustness verification, and SHAP analysis

The final evaluation uses the enabled base-learner subset, a linear-SVR meta-learner, 50 seeds (0–49), and 10-fold internal/external cross-validation.


In [ ]:
# ==============================================================================
# --- USER CONFIGURATION AREA ---
# ==============================================================================
# --- Evaluation protocol ---
N_SEEDS_FOR_EVALUATION = 50  # Use 50 independent partitions for SI-level robustness; reduce only for pilot runs.
N_SPLITS_OUTER_CV = 10       # Use 10 folds for stable estimates; use 5 when the dataset is very small.
N_SPLITS_INNER_CV = 10       # OOF folds for the meta-learner; keep below the sample count in each outer train set.

# --- SVR meta-learner tuning ---
META_LEARNER_N_ITER_BAYESIAN = 50  # Larger budgets search more SVR settings; reduce for exploratory runs.
META_LEARNER_N_SPLITS_CV = 10      # Use 10 folds when sample size permits; use 5 for small datasets.
META_LEARNER_SCORING = "r2"        # Optimize R^2; retain this metric for SI-consistent model selection.
META_TUNING_SEED = 0               # Fix the OOF partition for reproducible SVR tuning.
BAYES_OPTIMIZER_KWARGS = {"base_estimator": "GP", "acq_func": "EI"}  # GP/EI balances exploration and exploitation.

# --- Linear SVR search space ---
from skopt.space import Real
SVR_C_SEARCH_SPACE = Real(1e-3, 1e3, prior="log-uniform")  # Increase C when underfitting; decrease it when variance is high.
SVR_EPSILON_SEARCH_SPACE = Real(1e-4, 1.0, prior="log-uniform")  # Increase epsilon for noisy targets; decrease it for finer residual fitting.

# --- Output and visualization ---
OUTPUT_EXCEL_FILENAME = "il_lumo_shap_analysis_results.xlsx"  # Output workbook; change only to avoid overwriting a prior run.
N_FEATURES_TO_PLOT = 30              # Number of descriptors shown; lower for a compact figure.
PLOT_SHAP_SWARM_PLOT = True          # Set False for headless or faster execution.
SHAP_SWARM_SAMPLES_LIMIT = 5000      # Plot cap; lower when memory is limited, without changing exported SHAP values.

# --- Base-learner selection ---
# Keep all seven candidates enabled for general use; comment out entries only for a deliberate ablation.
ENABLED_BASE_LEARNERS = [
    "XGBR",  # XGBoost regression.
    "RF",    # Random Forest regression.
    "GBRT",  # Gradient Boosting regression.
    "HGBR",  # Histogram-based Gradient Boosting regression.
    "ETR",   # Extra-Trees regression.
    "CBR",   # CatBoost regression.
    "LGBM",  # LightGBM regression.
]
DEFAULT_MODEL_RANDOM_STATE = 0  # Keep fixed so stochastic base learners are comparable across runs.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as XGB
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import clone
from skopt import BayesSearchCV


if "X" not in globals() or "Y" not in globals() or "grid_searches" not in globals():
    raise RuntimeError("Run the hyperparameter-tuning cell first.")

filtered_grid_searches = {
    name: result for name, result in grid_searches.items()
    if name in ENABLED_BASE_LEARNERS and result is not None
}
model_names_available = list(filtered_grid_searches)
if not model_names_available:
    raise RuntimeError("No valid base learner is enabled.")
feature_names_list = list(X.columns)

def initialize_best_estimators(searches):
    return {
        name: clone(result.best_estimator_)
        for name, result in searches.items()
        if hasattr(result, "best_estimator_")
    }

def r2_to_weights(scores):
    values = np.asarray(list(scores), dtype=float)
    values[~np.isfinite(values)] = 0.0
    values = np.maximum(values, 0.0)
    total = values.sum()
    return values / total if total > 0 else np.ones(len(values)) / len(values)

def build_svr_pipeline():
    return Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="linear"))])

# The adjustable SVR dimensions are defined in the user configuration area above.
meta_learner_params = {
    "svr__C": SVR_C_SEARCH_SPACE,
    "svr__epsilon": SVR_EPSILON_SEARCH_SPACE,
}

def generate_oof_predictions(X_data, y_data, estimators, n_splits, cv_random_state):
    oof = np.zeros((len(X_data), len(estimators)))
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=cv_random_state)
    for model_idx, estimator in enumerate(estimators.values()):
        for train_idx, val_idx in cv.split(X_data, y_data):
            fitted = clone(estimator)
            fitted.fit(X_data.iloc[train_idx], y_data.iloc[train_idx])
            oof[val_idx, model_idx] = fitted.predict(X_data.iloc[val_idx])
    return oof

# Tune SVR once on OOF predictions, then reuse the parameters for all seeds.
base_estimators_for_meta = initialize_best_estimators(filtered_grid_searches)
oof_preds_full = generate_oof_predictions(
    X, Y, base_estimators_for_meta, META_LEARNER_N_SPLITS_CV, META_TUNING_SEED
)
meta_bayes_search = BayesSearchCV(
    estimator=build_svr_pipeline(),
    search_spaces=meta_learner_params,
    n_iter=META_LEARNER_N_ITER_BAYESIAN,
    scoring=META_LEARNER_SCORING,
    cv=KFold(
        n_splits=META_LEARNER_N_SPLITS_CV,
        shuffle=True,
        random_state=META_TUNING_SEED,
    ),
    n_jobs=-1,
    random_state=META_TUNING_SEED,
    optimizer_kwargs=BAYES_OPTIMIZER_KWARGS,
    verbose=0,
)
meta_bayes_search.fit(oof_preds_full, Y)
best_meta_learner_params = dict(meta_bayes_search.best_params_)
print(f"SVR meta-learner best CV R²: {meta_bayes_search.best_score_:.4f}")
print(f"SVR parameters: {best_meta_learner_params}")

def extract_tree_shap(estimator, X_val, model_name):
    if model_name == "XGBR":
        # XGBoost's native TreeSHAP avoids SHAP/XGBoost model-format incompatibilities.
        values = estimator.get_booster().predict(
            XGB.DMatrix(X_val), pred_contribs=True
        )[:, :-1]
        if values.shape[1] != len(feature_names_list):
            raise ValueError(f"Unexpected SHAP shape for {model_name}: {values.shape}")
        return values
    explainer = shap.TreeExplainer(estimator)
    raw = explainer.shap_values(X_val) if model_name == "CBR" else explainer(X_val).values
    values = np.asarray(raw)
    if values.ndim == 3:
        values = values[0]
    if values.ndim != 2 or values.shape[1] != len(feature_names_list):
        raise ValueError(f"Unexpected SHAP shape for {model_name}: {values.shape}")
    return values

all_fold_scores = []
seed_level_stack_abs = []
seed_level_base_abs = {name: [] for name in model_names_available}
seed_weight_records = []
base_sample_shap = {name: [] for name in model_names_available}
base_sample_x = {name: [] for name in model_names_available}
stack_sample_signed = []
stack_sample_x = []

for cv_random_state in range(N_SEEDS_FOR_EVALUATION):
    print(f"Seed {cv_random_state}/{N_SEEDS_FOR_EVALUATION - 1}")
    outer_cv = KFold(
        n_splits=N_SPLITS_OUTER_CV,
        shuffle=True,
        random_state=cv_random_state,
    )
    base_estimators = initialize_best_estimators(filtered_grid_searches)
    seed_payloads = []
    seed_base_r2 = {name: [] for name in model_names_available}

    for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X, Y), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = Y.iloc[train_idx], Y.iloc[val_idx]
        oof_preds_val = np.zeros((len(y_val), len(model_names_available)))
        fold_shap = {}

        for model_idx, (name, estimator) in enumerate(base_estimators.items()):
            fitted = clone(estimator)
            fitted.fit(X_train, y_train)
            pred = fitted.predict(X_val)
            oof_preds_val[:, model_idx] = pred
            r2_value = r2_score(y_val, pred)
            rmse_value = np.sqrt(mean_squared_error(y_val, pred))
            seed_base_r2[name].append(r2_value)
            all_fold_scores.extend([
                {"Seed": cv_random_state, "Fold": fold_idx, "Model": name, "Metric": "R2", "Value": r2_value},
                {"Seed": cv_random_state, "Fold": fold_idx, "Model": name, "Metric": "RMSE", "Value": rmse_value},
            ])
            try:
                values = extract_tree_shap(fitted, X_val, name)
                fold_shap[name] = values
                base_sample_shap[name].append(values)
                base_sample_x[name].append(X_val.copy())
            except Exception as exc:
                print(f"SHAP failed for {name}, seed={cv_random_state}, fold={fold_idx}: {exc}")

        # Inner folds create OOF predictions for the SVR meta-learner.
        oof_preds_train = np.zeros((len(X_train), len(model_names_available)))
        inner_cv = KFold(
            n_splits=N_SPLITS_INNER_CV,
            shuffle=True,
            random_state=cv_random_state,
        )
        for inner_train_idx, inner_val_idx in inner_cv.split(X_train, y_train):
            for model_idx, estimator in enumerate(base_estimators.values()):
                fitted_inner = clone(estimator)
                fitted_inner.fit(
                    X_train.iloc[inner_train_idx],
                    y_train.iloc[inner_train_idx],
                )
                oof_preds_train[inner_val_idx, model_idx] = fitted_inner.predict(
                    X_train.iloc[inner_val_idx]
                )

        meta_learner = build_svr_pipeline()
        meta_learner.set_params(**best_meta_learner_params)
        meta_learner.fit(oof_preds_train, y_train)
        stacking_pred = meta_learner.predict(oof_preds_val)
        stacking_r2 = r2_score(y_val, stacking_pred)
        stacking_rmse = np.sqrt(mean_squared_error(y_val, stacking_pred))
        all_fold_scores.extend([
            {"Seed": cv_random_state, "Fold": fold_idx, "Model": "Stacking", "Metric": "R2", "Value": stacking_r2},
            {"Seed": cv_random_state, "Fold": fold_idx, "Model": "Stacking", "Metric": "RMSE", "Value": stacking_rmse},
        ])
        seed_payloads.append({"X_val": X_val.copy(), "shap": fold_shap})

    # Use one R²-relative weight set per seed, then average seed-level vectors.
    seed_mean_r2 = {
        name: float(np.mean(seed_base_r2[name])) for name in model_names_available
    }
    weights = dict(zip(model_names_available, r2_to_weights(seed_mean_r2.values())))
    seed_weight_records.append({
        "Seed": cv_random_state,
        **{f"{name}_Mean_R2": seed_mean_r2[name] for name in model_names_available},
        **{f"{name}_SHAP_Weight": weights[name] for name in model_names_available},
    })

    seed_abs_parts = []
    seed_signed_parts = []
    seed_x_parts = []
    base_abs_parts = {name: [] for name in model_names_available}
    for payload in seed_payloads:
        if not payload["shap"]:
            continue
        signed = np.zeros_like(next(iter(payload["shap"].values())))
        absolute = np.zeros_like(signed)
        for name in model_names_available:
            if name not in payload["shap"]:
                continue
            values = payload["shap"][name]
            signed += weights[name] * values
            absolute += weights[name] * np.abs(values)
            base_abs_parts[name].append(np.abs(values))
        seed_signed_parts.append(signed)
        seed_abs_parts.append(absolute)
        seed_x_parts.append(payload["X_val"])

    if seed_abs_parts:
        seed_level_stack_abs.append(np.vstack(seed_abs_parts).mean(axis=0))
        for name in model_names_available:
            if base_abs_parts[name]:
                seed_level_base_abs[name].append(np.vstack(base_abs_parts[name]).mean(axis=0))
        stack_sample_signed.extend(seed_signed_parts)
        stack_sample_x.extend(seed_x_parts)

print("Aggregating metrics and SHAP results...")
final_analysis = {}
with pd.ExcelWriter(OUTPUT_EXCEL_FILENAME, engine="openpyxl") as writer:
    scores_df = pd.DataFrame(all_fold_scores)
    scores_pivot = scores_df.pivot_table(
        index=["Seed", "Fold"], columns=["Model", "Metric"], values="Value"
    )
    scores_pivot.columns = [f"{metric}_{model}" for model, metric in scores_pivot.columns]
    scores_pivot.to_excel(writer, sheet_name="Fold_Performance_Metrics")
    scores_df.groupby(["Seed", "Model", "Metric"], as_index=False)["Value"].mean().to_excel(
        writer, sheet_name="Seed_Performance_Summary", index=False
    )
    scores_df.groupby(["Model", "Metric"], as_index=False)["Value"].mean().pivot(
        index="Model", columns="Metric", values="Value"
    ).to_excel(writer, sheet_name="Overall_Performance_Summary")
    pd.DataFrame(seed_weight_records).to_excel(
        writer, sheet_name="Seed_SHAP_Weights", index=False
    )

    for name in model_names_available:
        if seed_level_base_abs[name]:
            values = np.vstack(seed_level_base_abs[name]).mean(axis=0)
            final_analysis[name] = {
                "global_importance": pd.Series(values, index=feature_names_list).sort_values(ascending=False)
            }
    if seed_level_stack_abs:
        values = np.vstack(seed_level_stack_abs).mean(axis=0)
        final_analysis["Stacking"] = {
            "global_importance": pd.Series(values, index=feature_names_list).sort_values(ascending=False)
        }

    plot_sources = {
        **{name: (base_sample_x[name], base_sample_shap[name]) for name in model_names_available},
        "Stacking": (stack_sample_x, stack_sample_signed),
    }
    for name, result in final_analysis.items():
        importance = result["global_importance"]
        importance.to_excel(writer, sheet_name=f"{name}_GlobalImportance")
        x_parts, shap_parts = plot_sources.get(name, ([], []))
        if not x_parts or not shap_parts:
            continue
        combined_x = pd.concat(x_parts, ignore_index=True)
        combined_shap = np.vstack(shap_parts)
        ordered = list(importance.index)
        export_df = pd.concat([
            combined_x.reset_index(drop=True),
            pd.DataFrame(combined_shap, columns=feature_names_list).add_prefix("SHAP_"),
        ], axis=1)
        export_df[ordered + [f"SHAP_{f}" for f in ordered]].to_excel(
            writer, sheet_name=f"{name}_SwarmPlotData", index=False
        )
        if PLOT_SHAP_SWARM_PLOT:
            limit = min(len(combined_x), SHAP_SWARM_SAMPLES_LIMIT) if SHAP_SWARM_SAMPLES_LIMIT else len(combined_x)
            top = ordered[:N_FEATURES_TO_PLOT]
            idx = [feature_names_list.index(f) for f in top]
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9))
            plt.sca(ax1)
            shap.summary_plot(
                combined_shap[:limit, idx],
                combined_x.iloc[:limit][top],
                feature_names=top,
                show=False,
            )
            ax1.set_title("SHAP Beeswarm Plot")
            top_series = importance.head(N_FEATURES_TO_PLOT).sort_values()
            sns.barplot(x=top_series.values, y=top_series.index, ax=ax2, orient="h", palette="viridis")
            ax2.set_title("Global Feature Importance")
            ax2.set_xlabel("R²-weighted mean(|SHAP|), arithmetic mean over seeds")
            ax2.set_ylabel("")
            fig.suptitle(
                f"SHAP Analysis for {name} (Seeds={N_SEEDS_FOR_EVALUATION}, Folds={N_SPLITS_OUTER_CV})"
            )
            plt.tight_layout()
            plt.show()

    if final_analysis:
        pd.concat(
            [item["global_importance"].rename(name) for name, item in final_analysis.items()],
            axis=1,
        ).sort_index().to_excel(writer, sheet_name="Global_Importance_Summary")

print(f"Finished. Seeds evaluated: 0..{N_SEEDS_FOR_EVALUATION - 1}")


# Prediction


In [ ]:
# ==============================================================================
# --- USER CONFIGURATION AREA ---
# ==============================================================================
# --- Prediction data ---
UNKNOWN_DATA_FILE = 'il_lumo_prediction_data.xlsx'  # Identifier first, then the 16 selected features in training order; omit LUMO.
# The prediction workbook must match the engineered feature order and omit the target column.
UNKNOWN_DATA_FILE_COLUMN_RANGE = (slice(None), slice(1, None))  # Select all rows and skip the identifier column.

# --- Full-data meta-learner training ---
REUSE_CACHED_META_PARAMS = True  # Reuse Cell 3 SVR settings; disable only to retune on the full-data OOF matrix.
PREDICT_CV_N_SPLITS = 10         # OOF folds for base learners; use 5 for very small datasets.
PREDICT_META_LEARNER_N_SPLITS_CV = 10   # Folds for fallback SVR tuning; reduce when data are scarce.
PREDICT_META_LEARNER_N_REPEATS_CV = 10  # Repeats improve fallback stability but increase runtime.
PREDICT_META_LEARNER_N_ITER_BAYESIAN = 50  # Search budget when no compatible cache is available.
BAYES_OPTIMIZER_KWARGS = {'base_estimator': 'GP', 'acq_func': 'EI'}  # GP/EI balances exploration and exploitation.

# --- Linear SVR search space (used only when the Cell 3 cache is unavailable) ---
from skopt.space import Real
SVR_C_SEARCH_SPACE = Real(1e-3, 1e3, prior='log-uniform')  # Increase C when underfitting; decrease it when variance is high.
SVR_EPSILON_SEARCH_SPACE = Real(1e-4, 1.0, prior='log-uniform')  # Increase epsilon for noisy targets; decrease it for finer residual fitting.

# --- Base-learner selection ---
# Keep all seven candidates enabled for general use; comment out entries only for a deliberate ablation.
ENABLED_BASE_LEARNERS = [
    'XGBR',  # XGBoost regression.
    'RF',    # Random Forest regression.
    'GBRT',  # Gradient Boosting regression.
    'HGBR',  # Histogram-based Gradient Boosting regression.
    'ETR',   # Extra-Trees regression.
    'CBR',   # CatBoost regression.
    'LGBM',  # LightGBM regression.
]

# --- Export ---
PREDICTION_OUTPUT_FILENAME_PREFIX = 'il_lumo_predictions'  # Project-specific prefix for prediction exports.
PREDICTION_EXPORT_TO_EXCEL = True  # Export the primary result workbook.
PREDICTION_EXPORT_TO_CSV = False   # Enable only when a CSV copy is required.

# --- Reproducibility ---
DEFAULT_MODEL_RANDOM_STATE = 0  # Keep aligned with the upstream cells for reproducible predictions.


# ==============================================================================
# --- MAIN SCRIPT EXECUTION (Do not modify this section) ---
# ==============================================================================

# --- 1. Library Imports and Environment Check ---
import pandas as pd
import numpy as np
import time
import os
import re
import sys 

import xgboost as XGB
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.base import clone
from skopt import BayesSearchCV

print("--- Initiating Prediction on Unknown Data ---")

try:
    from catboost import CatBoostRegressor
    catboost_available = True
except ImportError:
    catboost_available = False
try:
    from lightgbm import LGBMRegressor
    lightgbm_available = True
except ImportError:
    lightgbm_available = False
print("Library imports and environment check completed.")


# --- 2. Core Variable Loading and Preparation ---
# Check for the existence of required variables from upstream scripts.
if 'X' not in locals() and 'X' not in globals():
    print("Error: Variable 'X' (Training Features) is undefined. Please ensure the data preparation and optimization scripts have been executed.")
    sys.exit(1)
if 'Y' not in locals() and 'Y' not in globals():
    print("Error: Variable 'Y' (Training Targets) is undefined. Please ensure the data preparation and optimization scripts have been executed.")
    sys.exit(1)
if 'grid_searches' not in locals() and 'grid_searches' not in globals():
    print("Error: Variable 'grid_searches' is undefined. Please ensure the optimization script has been executed.")
    sys.exit(1)

# Filter the optimized models based on the current configuration.
filtered_grid_searches = {name: gs for name, gs in grid_searches.items() if name in ENABLED_BASE_LEARNERS}
print(f"\nModel filtering completed: Selected {len(filtered_grid_searches)} enabled models from {len(grid_searches)} available: {list(filtered_grid_searches.keys())}")
if not filtered_grid_searches:
    print("\n!!! FATAL ERROR: No Base Learners selected. Please check the ENABLED_BASE_LEARNERS list in the configuration.")
    sys.exit(1)

# Sanitize feature names in the training set to prevent errors in models like XGBoost.
# Store the sanitized list to ensure alignment with the prediction dataset.
feature_names_list = list(X.columns)
X.columns = [re.sub(r'\[|\]|<', '_', col) for col in feature_names_list]
feature_names_list = list(X.columns) # Update the list to reflect sanitized names.

print(f"\nAttempting to load unknown prediction dataset from '{UNKNOWN_DATA_FILE}'...")
if not os.path.exists(UNKNOWN_DATA_FILE):
    print(f"Error: Prediction file '{UNKNOWN_DATA_FILE}' does not exist. Please verify the file path.")
    sys.exit(1) 
else:
    try:
        full_data_from_excel = pd.read_excel(UNKNOWN_DATA_FILE)
        # Apply slicing based on UNKNOWN_DATA_FILE_COLUMN_RANGE
        X_new = full_data_from_excel.iloc[UNKNOWN_DATA_FILE_COLUMN_RANGE].copy()
    except Exception as e:
        print(f"Error: Failed to load or process '{UNKNOWN_DATA_FILE}': {e}. Please check file content and column range settings.")
        sys.exit(1) 

original_X_new_index = X_new.index # Preserve original indices for result export.

# --------------------------------------------------------------------------------
# --- Column Alignment Logic: Positional Mapping ---
# --------------------------------------------------------------------------------
# Verify that the number of features in the prediction set matches the training set.
if X_new.shape[1] != len(feature_names_list):
    print(f"Error: Prediction set X_new has {X_new.shape[1]} columns, but training set has {len(feature_names_list)} columns. Feature count mismatch.")
    print("Please verify 'UNKNOWN_DATA_FILE_COLUMN_RANGE' to ensure the extracted feature count matches the training data.")
    sys.exit(1)

# Directly map training feature names to the prediction dataset columns by position.
# This assumes the column order in the new dataset exactly matches the training data.
# This prevents issues arising from minor naming discrepancies.
X_new.columns = feature_names_list

# Check for missing values in the raw prediction data.
if X_new.isnull().any().any():
    missing_cols_in_raw_X_new = X_new.columns[X_new.isnull().any()].tolist()
    print(f"Warning: Prediction dataset X_new contains missing values (NaN). Affected columns: {missing_cols_in_raw_X_new}")
    # X_new = X_new.fillna(0) # Optional: Uncomment to enable zero-filling if required.
# --------------------------------------------------------------------------------

print(f"\nTraining Data X shape: {X.shape}")
print(f"Prediction Data X_new shape (after column alignment): {X_new.shape}")


# --- 3. Core Functions and Model Training ---
def initialize_best_estimators(grid_searches_dict):
    estimators_init = {}
    available_models = {
        'XGBR': (XGB.XGBRegressor, {'objective': 'reg:squarederror', 'random_state': DEFAULT_MODEL_RANDOM_STATE}),
        'RF': (RandomForestRegressor, {'random_state': DEFAULT_MODEL_RANDOM_STATE}),
        'GBRT': (GradientBoostingRegressor, {'random_state': DEFAULT_MODEL_RANDOM_STATE}),
        'ETR': (ExtraTreesRegressor, {'random_state': DEFAULT_MODEL_RANDOM_STATE}),
        'HGBR': (HistGradientBoostingRegressor, {'random_state': DEFAULT_MODEL_RANDOM_STATE})
    }
    if catboost_available: available_models['CBR'] = (CatBoostRegressor, {'verbose': False, 'random_state': DEFAULT_MODEL_RANDOM_STATE, 'allow_writing_files': False})
    if lightgbm_available: available_models['LGBM'] = (LGBMRegressor, {'random_state': DEFAULT_MODEL_RANDOM_STATE, 'verbosity': -1, 'objective': 'regression'})

    print("Initializing Base Learners...")
    for name in grid_searches_dict.keys():
        if name not in available_models:
            print(f"  Warning: Model {name} is not in the available models list, skipping initialization.")
            continue
        model_class, fixed_params = available_models[name]
        if name in grid_searches_dict and grid_searches_dict[name] is not None and hasattr(grid_searches_dict[name], 'best_estimator_'):
            try:
                estimators_init[name] = grid_searches_dict[name].best_estimator_
                print(f"  Successfully initialized {name} using optimized parameters.")
            except Exception as e:
                print(f"  Error initializing {name} (from best_estimator_): {e}. Skipping this model.")
        else:
            print(f"  Warning: Optimization results for {name} not found. Attempting initialization with default parameters.")
            try:
                estimators_init[name] = model_class(**fixed_params)
                print(f"  Successfully initialized {name} using default parameters.")
            except Exception as e:
                print(f"  Error initializing {name} (using defaults): {e}. Skipping this model.")
    initialized_estimators = {k: v for k, v in estimators_init.items() if v is not None}
    if not initialized_estimators: print("Warning: No models were successfully initialized!")
    return initialized_estimators

# Linear SVR search space; kernel-specific nonlinear parameters are intentionally excluded.
# The adjustable SVR dimensions are defined in the user configuration area above.
meta_learner_params = {
    'svr__C': SVR_C_SEARCH_SPACE,
    'svr__epsilon': SVR_EPSILON_SEARCH_SPACE
}
final_meta_learner = None
base_estimators_for_final_training = None

print("\n--- Training the final Stacked Model on the full dataset ---")
base_estimators_for_oof = initialize_best_estimators(filtered_grid_searches)
if not base_estimators_for_oof:
    print("Error: No Base Learners available for Stacking Model training.")
    sys.exit(1)

model_names_available = list(base_estimators_for_oof.keys())
oof_preds_for_meta_training = np.zeros((len(X), len(model_names_available)))
kf_for_oof = KFold(
    n_splits=PREDICT_CV_N_SPLITS,
    shuffle=True,
    random_state=DEFAULT_MODEL_RANDOM_STATE,
)

print(f"Starting {PREDICT_CV_N_SPLITS}-fold OOF generation (seed={DEFAULT_MODEL_RANDOM_STATE})...")
for fold_idx, (train_idx, val_idx) in enumerate(kf_for_oof.split(X, Y), start=1):
    print(f"  Fold {fold_idx}/{PREDICT_CV_N_SPLITS}")
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold = Y.iloc[train_idx]
    for model_idx, estimator_template in enumerate(base_estimators_for_oof.values()):
        estimator_fold = clone(estimator_template)
        try:
            estimator_fold.fit(X_train_fold, y_train_fold)
            oof_preds_for_meta_training[val_idx, model_idx] = estimator_fold.predict(X_val_fold)
        except Exception as exc:
            print(f"    Warning: OOF fit failed for model index {model_idx}: {exc}")

if np.all(oof_preds_for_meta_training == 0) or np.any(np.isnan(oof_preds_for_meta_training)):
    print("Error: Invalid OOF predictions; SVR training cannot proceed.")
    sys.exit(1)

cached_meta_params = None
if REUSE_CACHED_META_PARAMS:
    candidate = globals().get('best_meta_learner_params')
    if candidate is None and 'meta_bayes_search' in globals():
        candidate = getattr(meta_bayes_search, 'best_params_', None)
    if isinstance(candidate, dict) and set(candidate) == set(meta_learner_params):
        cached_meta_params = {key: candidate[key] for key in meta_learner_params}
        print("Reusing compatible SVR hyperparameters from the evaluation cell.")

if cached_meta_params is None:
    print("Tuning SVR hyperparameters on the full-data OOF matrix...")
    meta_bayes_search = BayesSearchCV(
        estimator=Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='linear'))]),
        search_spaces=meta_learner_params,
        n_iter=PREDICT_META_LEARNER_N_ITER_BAYESIAN,
        scoring='r2',
        cv=RepeatedKFold(
            n_splits=PREDICT_META_LEARNER_N_SPLITS_CV,
            n_repeats=PREDICT_META_LEARNER_N_REPEATS_CV,
            random_state=DEFAULT_MODEL_RANDOM_STATE,
        ),
        n_jobs=-1,
        random_state=DEFAULT_MODEL_RANDOM_STATE,
        optimizer_kwargs=BAYES_OPTIMIZER_KWARGS,
        verbose=1,
    )
    meta_bayes_search.fit(oof_preds_for_meta_training, Y)
    best_meta_learner_params = dict(meta_bayes_search.best_params_)
else:
    best_meta_learner_params = cached_meta_params

final_meta_learner = Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='linear'))])
final_meta_learner.set_params(**best_meta_learner_params)
final_meta_learner.fit(oof_preds_for_meta_training, Y)
print(f"SVR parameters used for full-data prediction: {best_meta_learner_params}")

print("\n--- Fitting final base learners on the full dataset ---")
base_estimators_for_final_training = initialize_best_estimators(filtered_grid_searches)
for name, estimator in base_estimators_for_final_training.items():
    try:
        estimator.fit(X, Y)
    except Exception as exc:
        print(f"Warning: Final fit failed for {name}: {exc}")
        base_estimators_for_final_training[name] = None
base_estimators_for_final_training = {
    name: estimator
    for name, estimator in base_estimators_for_final_training.items()
    if estimator is not None
}

# --- 4. Prediction on Unknown Data ---
if not final_meta_learner or not base_estimators_for_final_training:
    print("FATAL ERROR: Stacking Model construction failed. Unable to proceed with prediction."); sys.exit(1)

print("\n--- Performing Stacked Prediction on Unknown Data ---")
base_predictions_on_new_data, base_model_names_for_prediction = [], []
print("Generating Base Learner predictions on unknown data...")
for name, estimator in base_estimators_for_final_training.items():
    try:
        pred = estimator.predict(X_new)
        base_predictions_on_new_data.append(pred)
        base_model_names_for_prediction.append(name)
        print(f"  {name} prediction completed.")
    except Exception as e:
        print(f"  Error: Model {name} failed during prediction on unknown data: {e}. Skipping this model.")

if not base_predictions_on_new_data:
    print("Error: All Base Learners failed to predict on unknown data."); sys.exit(1)

meta_features_for_prediction = np.column_stack(base_predictions_on_new_data)
print(f"Base Learner Prediction Shape (Meta-Learner Input): {meta_features_for_prediction.shape}")

print("Executing final Stacked Prediction using Meta-Learner...")
try:
    final_stacked_predictions = final_meta_learner.predict(meta_features_for_prediction)
    print("Final Stacked Prediction completed.")
except Exception as e:
    print(f"Error during Meta-Learner final prediction: {e}"); sys.exit(1)


# --- 5. Result Compilation and Export ---
print("\n--- Compiling and Exporting Prediction Results ---")
results_df = pd.DataFrame(index=original_X_new_index) # Use original indices
for i, name in enumerate(base_model_names_for_prediction):
    results_df[f'{name}_Prediction'] = base_predictions_on_new_data[i]
results_df['Stacked_Prediction'] = final_stacked_predictions

current_timestamp = time.strftime("%Y%m%d_%H%M%S")
output_filename_base = f"{PREDICTION_OUTPUT_FILENAME_PREFIX}_{current_timestamp}"

if PREDICTION_EXPORT_TO_EXCEL:
    excel_filename = f"{output_filename_base}.xlsx"
    try:
        results_df.to_excel(excel_filename, index=True, engine='openpyxl')
        print(f"Prediction results successfully exported to Excel: {excel_filename}")
    except ImportError:
        print("!!! 'openpyxl' library required for Excel export. Falling back to CSV export.")
        PREDICTION_EXPORT_TO_EXCEL = False
    except Exception as e:
        print(f"!!! Error exporting results to Excel: {e}")
        PREDICTION_EXPORT_TO_EXCEL = False

if PREDICTION_EXPORT_TO_CSV or not PREDICTION_EXPORT_TO_EXCEL:
    csv_filename = f"{output_filename_base}.csv"
    try:
        results_df.to_csv(csv_filename, index=True)
        print(f"Prediction results successfully exported to CSV: {csv_filename}")
    except Exception as e:
        print(f"!!! Error exporting results to CSV: {e}")

print("\nUnknown Data Prediction Script Execution Finished.")
